In [26]:
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client.session")
warnings.filterwarnings("ignore", message="datetime.datetime.utcnow")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Instalasi Unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

# Instalasi Library RAG
!pip install -qU langchain langchain-community langchain-core langchain-text-splitters
!pip install -qU pypdf chromadb sentence-transformers duckduckgo-search rank_bm25 langchain-huggingface

print("[INFO] Semua library RAG berhasil diinstall!")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-i8k5lb3n/unsloth_85923f4368f04dd3a5d9a6e611a80764
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-i8k5lb3n/unsloth_85923f4368f04dd3a5d9a6e611a80764
  Resolved https://github.com/unslothai/unsloth.git to commit 2aef1a23cb92922cb22d3e484b1e67f241c31c3e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 63.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 29.2 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xformers

In [14]:
!pip uninstall -y torchcodec

Found existing installation: torchcodec 0.11.0+cu128
Uninstalling torchcodec-0.11.0+cu128:
  Successfully uninstalled torchcodec-0.11.0+cu128


In [16]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load & Chungking Dokumen PDF
pdf_folder_path = "/content/drive/MyDrive/Legal-RAG-Dataset"

print(f"[INFO] Membaca dokumen PDF dari: {pdf_folder_path}")

loader = PyPDFDirectoryLoader(pdf_folder_path)
documents = loader.load()

print(f"[INFO] Selesai membaca! Total halaman yang ditemukan: {len(documents)} halaman.")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
    length_function = len,
)

chunks = text_splitter.split_documents(documents)

print(f"[INFO] Dokumen berhasil dipotong menjadi {len(chunks)} chunks.")
print("=" * 50)
print("[INFO] Melihat Chunk Pertama:")
print(chunks[0].page_content)

[INFO] Membaca dokumen PDF dari: /content/drive/MyDrive/Legal-RAG-Dataset
[INFO] Selesai membaca! Total halaman yang ditemukan: 1949 halaman.
[INFO] Dokumen berhasil dipotong menjadi 3242 chunks.
[INFO] Melihat Chunk Pertama:
SALINAN
PRESIDEN
REPUBLTI( INDONESIA
PERATURAN PEMERINTAH REPUBLIK INDONESIA
NOMOR 51 TAHUN 2023
TENTANG
PERUBAHAN ATAS PERATURAN PEMERINTAH NOMOR 36 TAHUN 2O21
TENTANG PENGUPAHAN
DENGAN RAHMAT TUHAN YANG MAHA ESA
PRESIDEN REPUBLIK INDONESIA,
Menimbang a. bahwa untuk melaksanakan ketentuan Pasal 88C dan
Pasal 88D dalam Pasal 81 angka 28 Undang-Undang
Nomor 6 Tahun 2023 tentang Penetapan Peraturan
Pemerintah Pengganti Undang-Undang Nomor 2
Tahun 2022 tentang Cipta Kerja Menjadi Undang-Undang
perlu dilakukan perubahan mengenai ketentuan
Upah minimum;
b. bahwa untuk menjaga daya beli Pekerja/Buruh dan
stabilitas ekonomi nasional serta memperhatikan dinamika
perkembang€rn hubungan industrial, ketentuan mengenai
Upah minimum sebagaimana telah diatur dalam
Peraturan Pem

In [17]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Embedding & Vector Database
print("[INFO] Mengunduh model Embedding Multilingual... (sabarr!)")

# inisialisasi model embedding open-source untuk bahasa indonesia
embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("[INFO] Mengubah chunks menjadi vektor dan menyimpannya di ChromaDB...")
# membuat Vector Database lokal menggunakan Chroma
vectorstore = Chroma.from_documents(
    documents=chunks,\
    embedding=embedding_model,
    persist_directory="./chroma_db_legal"
)

# membuat Retriever untuk mengambil 3 dokumen paling mirip
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("[INFO] Vector Database berhasil dibuat dan siap digunakan!")
print("=" * 50)

# ujicoba pencarian
pertanyaan_tes = "Apa yang dimaksud dengan Perizinan Berusaha Berbasis Risiko?"
print(f"[INFO] Menguji Pencarian dengan Pertanyaan: '{pertanyaan_tes}'\n")

hasil_pencarian = retriever.invoke(pertanyaan_tes)

for i, doc in enumerate(hasil_pencarian):
    print(f"=== Dokumen Relevan Ke-{i+1} ===")
    print(doc.page_content)
    print()


[INFO] Mengunduh model Embedding Multilingual... (sabarr!)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[INFO] Mengubah chunks menjadi vektor dan menyimpannya di ChromaDB...
[INFO] Vector Database berhasil dibuat dan siap digunakan!
[INFO] Menguji Pencarian dengan Pertanyaan: 'Apa yang dimaksud dengan Perizinan Berusaha Berbasis Risiko?'

=== Dokumen Relevan Ke-1 ===
2021, No.15 -9- 
BAB II 
PENGATURAN PERIZINAN BERUSAHA BERBASIS RISIKO 
 
Bagian Kesatu 
Analisis Risiko 
 
Pasal 7 
(1) Perizinan Berusaha Berbasis Risiko dilakukan 
berdasarkan penetapan tingkat Risiko dan peringka t 
skala kegiatan usaha meliputi UMK- M dan/atau 
usaha besar. 
(2) Penetapan tingkat Risiko sebagaimana dimaksud pada 
ayat (1) dilakukan berdasarkan hasil analisis Risiko.  
(3) Analisis Risiko sebagaimana dimaksud pada ayat (2) 
wajib dilakukan secara transparan, akuntabel, dan 
mengedepankan prinsip kehati-hatian berdasarkan 
data dan/atau penilaian profesional. 
(4) Tingkat Risiko sebagaimana dimaksud pada ayat (2) 
menentukan jenis Perizinan Berusaha. 
 
Pasal 8 
Pelaksanaan analisis Risiko sebagaimana dim

In [18]:
# memuat model & membuat Chatbot RAG
from unsloth import FastLanguageModel
from IPython.display import display, Markdown
import torch

# panggil model dari HF
model_name = "Saranjana/Llama-3-8b-Legal-RAG"

print(f"[INFO] Sedang memanggil model {model_name} dari Hugging Face...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)

# template prompt
alpaca_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

Kamu adalah asisten AI dari Tim Legal yang bertugas menjawab pertanyaan pengguna secara jelas berdasarkan konteks dokumen hukum yang diberikan.

Konteks Dokumen:
{}<|eot_id|><|start_header_id|>user<|end_header_id|>

{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

# fungsi utama untuk RAG
def jawab_pertanyaan_legal(pertanyaan):
    # 1. ambil dokumen yg relevan
    docs = retriever.invoke(pertanyaan)
    konteks_gabungan = "\n\n".join([doc.page_content for doc in docs])

    # 2. masukkan ke dalam prompt
    prompt_lengkap = alpaca_prompt.format(konteks_gabungan, pertanyaan)
    inputs = tokenizer([prompt_lengkap], return_tensors="pt").to("cuda")

    # 3. model menjawab
    outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True)
    jawaban_kotor = tokenizer.decode(outputs[0], skip_special_tokens=False)

    # Pisahkan teks tepat di titik di mana assistant mulai berbicara
    penanda = "<|start_header_id|>assistant<|end_header_id|>"
    if penanda in jawaban_kotor:
        jawaban_bersih = jawaban_kotor.split(penanda)[-1]
    else:
        jawaban_bersih = jawaban_kotor.split("assistant")[-1]

    # 4. membersihkan jawaban (hanya ambil teks setelah bagian assistant)
    jawaban_bersih = jawaban_bersih.replace("<|eot_id|>", "").strip()
    return jawaban_bersih

print("\n" + "="*50)
print("[INFO] Chatbot Siap Digunakan!")
print("Ketik 'Keluar' atau 'Exit' untuk menghentikan percakapan!")
print("="*50 + "\n")

while True:
    pertanyaan_user = input("Pertanyaan Legal: ")

    # Kondisi untuk keluar dari loop
    if pertanyaan_user.lower() in ['keluar', 'exit', 'quit']:
        print("Sesi ditutup. Terimakasih!")
        break

    print("Membaca undang-undang dan berpikir...\n")

    # memanggil fungsi RAG
    jawaban_bot = jawab_pertanyaan_legal(pertanyaan_user)

    display(Markdown(f"** Jawaban:**\n{jawaban_bot}"))
    print("=" * 50)

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[INFO] Sedang memanggil model Saranjana/Llama-3-8b-Legal-RAG dari Hugging Face...
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

Pertanyaan Legal: Apa perbedaan antara Perizinan Berusaha dengan Perizinan Berusaha Untuk Menunjang Kegiatan Usaha?


Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Membaca undang-undang dan berpikir...



** Jawaban:**
Perbedaan antara Perizinan Berusaha dan Perizinan Berusaha Untuk Menunjang Kegiatan Usaha terletak pada tujuan dan tujuan dari peraturan-peraturan ini. 

Perizinan Berusaha adalah peraturan yang bertujuan untuk mengatur dan mengawasi kegiatan usaha, memastikan bahwa usaha dilakukan sesuai dengan hukum dan peraturan yang berlaku. Tujuan utamanya adalah untuk memastikan bahwa usaha dilakukan dengan aman dan beretika, dengan memenuhi persyaratan dan kewajiban yang ditetapkan oleh peraturan.

Di sisi lain, Perizinan Berusaha Untuk Menunjang Kegiatan Usaha adalah peraturan yang bertujuan untuk memberikan perlindungan dan keamanan bagi usaha dan pekerja. Tujuan utamanya adalah untuk memastikan bahwa usaha dilakukan dengan baik, dengan memenuhi persyaratan dan kewajiban yang ditetapkan oleh peraturan, dan untuk memberikan perlindungan dan keamanan bagi pekerja dan usaha.

Secara ringkas, Perizinan Berusaha adalah peraturan yang bertujuan untuk mengatur dan mengawasi kegiatan usaha, sedangkan Perizinan Berusaha Untuk Menunjang Kegiatan Usaha adalah peraturan yang bertujuan untuk memberikan perlindungan dan keamanan bagi usaha dan pekerja.

Pertanyaan Legal: exit
Sesi ditutup. Terimakasih!


In [19]:
# ensemble retriever
from langchain_community.retrievers import BM25Retriever

print("[INFO] Membangun Ensemble Retriever...")

# 1. siapkan BM25 (pencari kata kunci)
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

# 2. siapkan Vector Store (pencari makna)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 3. gabungkan keduanya (50:50)
def custom_ensemble_retriever(pertanyaan, bobot_bm25=0.5, bobot_vector=0.5):
    hasil_bm25 = bm25_retriever.invoke(pertanyaan)
    hasil_vector = vector_retriever.invoke(pertanyaan)

    dokumen_gabungan = []
    teks_tersimpan = set()

    daftar_awal = hasil_vector + hasil_bm25 if bobot_vector >= bobot_bm25 else hasil_bm25 + hasil_vector

    for doc in daftar_awal:
        if doc.page_content not in teks_tersimpan:
            teks_tersimpan.add(doc.page_content)
            dokumen_gabungan.append(doc)

    return dokumen_gabungan[:5]

print("[INFO] Ensemble Retriever berhasil dibuat!")


[INFO] Membangun Ensemble Retriever...
[INFO] Ensemble Retriever berhasil dibuat!


In [20]:
# Chatbot RAG
def bot_jawab_pertanyaan(pertanyaan):
    docs = custom_ensemble_retriever(pertanyaan, bobot_bm25=0.5, bobot_vector=0.5)

    sumber_referensi = []
    konteks_list = []

    for i, doc in enumerate(docs):
        konteks_list.append(f"Dokumen {i+1}:\n{doc.page_content}")

        source_name = doc.metadata.get('source', 'Dokumen Tidak Diketahui').split('/')[-1]
        page_num = doc.metadata.get('page', 'Tidak Diketahui')
        sumber_referensi.append(f"- {source_name} (Halaman {page_num})")

    konteks_gabungan = "\n\n".join(konteks_list)

    prompt_lengkap = alpaca_prompt.format(konteks_gabungan, pertanyaan)
    inputs = tokenizer([prompt_lengkap], return_tensors="pt").to("cuda")

    outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True)

    jawaban_kotor = tokenizer.decode(outputs[0], skip_special_tokens=False)
    penanda = "<|start_header_id|>assistant<|end_header_id|>"

    if penanda in jawaban_kotor:
        jawaban_bersih = jawaban_kotor.split(penanda)[-1]
    else:
        jawaban_bersih = jawaban_kotor.split(assistant)[-1]

    jawaban_bersih = jawaban_bersih.replace("<|eot_id|>", "").strip()

    referensi_unik = list(set(sumber_referensi))
    teks_sumber = "\n".join(referensi_unik)

    hasil_akhir = f"{jawaban_bersih}\n\n** Sumber Referensi: **\n{teks_sumber}"
    return hasil_akhir

# Dashboard
print("\n" + "="*50)
print("[INFO] Chatbot Tim Legal Siap!")
print("Ketik 'keluar' atau 'exit' untuk menghentikan percakapan!")
print("="*50 + "\n")

while True:
    pertanyaan_user = input("Pertanyaan : ")

    if pertanyaan_user.lower() in ['keluar', 'exit', 'quit']:
        print("Sesi ditutup. Terima kasih!")
        break

    print("Berpikir...\n")

    jawaban_bot = bot_jawab_pertanyaan(pertanyaan_user)

    display(Markdown(f"**Jawaban:**\n{jawaban_bot}"))
    print("=" * 50)


[INFO] Chatbot Tim Legal Siap!
Ketik 'keluar' atau 'exit' untuk menghentikan percakapan!

Pertanyaan : Apa saja hak staf admin terkait upah lembur?


Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Berpikir...



**Jawaban:**
Hak staf admin terkait upah lembur dapat berbeda-beda tergantung pada perusahaan, lokasi, dan peraturan lokal. Biasanya, staf admin diberikan upah lembur sebagai kompensasi tambahan untuk jam kerja yang melebihi jam kerja normal. Beberapa hak staf admin terkait upah lembur termasuk:

1. Upah lembur: Staf admin menerima upah lembur untuk setiap jam kerja yang melebihi jam kerja normal.

2. Jam kerja normal: Jam kerja normal dapat bervariasi antara perusahaan, tetapi biasanya antara 40-50 jam per minggu.

3. Perhitungan upah lembur: Perusahaan mungkin menggunakan berbagai metode untuk menghitung upah lembur, seperti jam, satuan waktu, atau satuan hasil.

4. Pemberitahuan: Staf admin harus diberitahu secara tepat waktu jika mereka akan diberikan upah lembur.

5. Penerimaan: Staf admin harus menerima upah lembur secara tepat waktu dan sesuai dengan kesepakatan yang telah disepakati.

6. Penghitungan upah lembur: Perusahaan harus menghitung upah lembur secara akurat dan sesuai dengan kesepakatan yang telah disepakati.

7. Denda: Staf admin dapat dikenakan denda jika mereka tidak melaksanakan jam kerja normal atau tidak melaksanakan tugas-tugas mereka dengan benar.

8. Pengadilan: Staf admin dapat mengajukan pengadilan jika mereka tidak setuju dengan upah lembur yang diberikan.

9. Kesepakatan: Staf admin dan perusahaan harus mencapai kesepakatan tentang upah lembur sebelum staf admin mulai bekerja.

10. Penyimpanan: Staf admin harus menyimpan bukti-bukti yang mendukung klaim mereka atas upah lembur.

11. Periode pembayaran: Staf admin harus menerima pembayaran upah lembur secara teratur, biasanya setiap bulan.

12. Pembayaran minimum: Staf admin harus menerima pembayaran minimum untuk upah lembur, sesuai dengan peraturan perundang-und

** Sumber Referensi: **
- UU Nomor 6 Tahun 2023.pdf (Halaman 977)
- PP Nomor 35 Tahun 2021.pdf (Halaman 4)
- UU Nomor 6 Tahun 2023.pdf (Halaman 723)
- UU Nomor 6 Tahun 2023.pdf (Halaman 561)
- UU Nomor 6 Tahun 2023.pdf (Halaman 173)

Pertanyaan : exit
Sesi ditutup. Terima kasih!


In [23]:
import warnings
warnings.filterwarnings("ignore")

from sentence_transformers import CrossEncoder
from duckduckgo_search import DDGS
import numpy as np

print("[INFO] Memuat Model Reranker (Cross-Encoder)...")

# inisialisasi model reranker menggunkan BAAI/bge-reranker-base
reranker_model = CrossEncoder("BAAI/bge-reranker-base")

# inisialisasi fitur web search fallback
def pencarian_web_mandiri(query):
    try:
        # Mengambil 3 hasil pencarian teratas dari DuckDuckGo
        hasil = DDGS().text(query, max_results=3)
        teks_hasil = "\n\n".join([f"Konteks Web: {item['body']}" for item in hasil])
        return teks_hasil
    except Exception as e:
        return f"Gagal mengakses internet: {e}"

def generate_hypothetical_answers(pertanyaan):

    prompt_hyde = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Kamu adalah pakar hukum. Berikan contoh jawaban singkat (bisa berupa halusinasi akurat) untuk pertanyaan pengguna berikut agar membantu pencarian dokumen. Jangan menulis instruksi lain, langsung berikan draf jawaban.
<|eot_id|><|start_header_id|>user<|end_header_id|>
{pertanyaan}
<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

    inputs = tokenizer([prompt_hyde], return_tensors="pt").to("cuda")

    # buat 2 verifikasi jawaban menggunakan num_return_sequences dan do_sample
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        num_return_sequences=2,
        do_sample=True,
        temperature=0.7,
        use_cache=True
    )

    jawaban_hyde = []
    input_length = inputs["input_ids"].shape[1]
    for output in outputs:
        text = tokenizer.decode(output[input_length:], skip_special_tokens=True).strip()
        jawaban_hyde.append(text)

    return jawaban_hyde

# RAG Pipeline
def advanced_rag_pipeline(pertanyaan, threshold_skor=0.1):
    print("[HyDE] Menghasilkan jawaban hipotesisi untuk memperluas pencarian...")
    jawaban_fiktif = generate_hypothetical_answers(pertanyaan)

    # mengumpulkan kandidat dokumen dari pertanyaan asli * 2 jawaban hipotesis
    pool_kandidat_dokumen = []
    teks_tersimpan = set()

    for query_pencarian in [pertanyaan] + jawaban_fiktif:
        docs = custom_ensemble_retriever(query_pencarian)
        for doc in docs:
            if doc.page_content not in teks_tersimpan:
                teks_tersimpan.add(doc.page_content)
                pool_kandidat_dokumen.append(doc)

    if not pool_kandidat_dokumen:
        print("[INFO] Tidak ada dokumen lokal yang ditemukan. Beralih ke Web Search!")
        hasil_web = pencarian_web_mandiri(pertanyaan)
        return hasil_web, ["Hasil Pencarian Web (DuckDuckGo)"]

    # reranking (Cross-Encoder)
    print(f"[Reranker] Mengevaluasi {len(pool_kandidat_dokumen)} kandidat dokumen...")
    pairs = [[pertanyaan, doc.page_content] for doc in pool_kandidat_dokumen]

    skor_reranker = reranker_model.predict(pairs)
    indeks_terurut = np.argsort(skor_reranker)[::-1]

    skor_top_1 = skor_reranker[indeks_terurut[0]]
    print(f"[INFO] Skor Reranker Top-1: {skor_top_1:.4f}")

    # logika kondisioanal (Treshold Fallback)
    if skor_top_1 < threshold_skor:
        print(f"[INFO] Skor di bawah threshold ({threshold_skor}). Mengabaikan dokumen lokal dan beralih ke Internet...")
        hasil_web = pencarian_web_mandiri(pertanyaan)
        konteks_web = f"Informasi Real-time dari Internet:\n{hasil_web}"
        prompt_lengkap = alpaca_prompt.format(konteks_web, pertanyaan)
        inputs = tokenizer([prompt_lengkap], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True)

        jawaban_kotor = tokenizer.decode(outputs[0], skip_special_tokens=False)
        penanda = "<|start_header_id|>assistant<|end_header_id|>"
        jawaban_bersih = jawaban_kotor.split(penanda)[-1] if penanda in jawaban_kotor else jawaban_kotor.split("assistant")[-1]
        jawaban_bersih = jawaban_bersih.replace("<|eot_id|>", "").strip()

        return f"*(Diambil dari Live Web Search)*\n{jawaban_bersih}", ["Pencarian Langsung Internet (DuckDuckGo API)"]

    dokumen_terpilih = [pool_kandidat_dokumen[idx] for idx in indeks_terurut[:3]]

    konteks_list = []
    sumber_referensi = []
    for i, doc in enumerate(dokumen_terpilih):
        konteks_list.append(f"Dokumen {i+1}:\n{doc.page_content}")
        source_name = doc.metadata.get('source', 'Dokumen').split('/')[-1]
        page_num = doc.metadata.get('page', 'Tidak Diketahui')
        sumber_referensi.append(f"- {source_name} (Halaman {page_num}) [Reranker Score: {skor_reranker[indeks_terurut[i]]:.2f}]")

    konteks_gabungan = "\n\n".join(konteks_list)

    prompt_lengkap = alpaca_prompt.format(konteks_gabungan, pertanyaan)
    inputs = tokenizer([prompt_lengkap], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True)

    jawaban_kotor = tokenizer.decode(outputs[0], skip_special_tokens=False)
    penanda = "<|start_header_id|>assistant<|end_header_id|>"
    if penanda in jawaban_kotor:
        jawaban_bersih = jawaban_kotor.split(penanda)[-1]
    else:
        jawaban_bersih = jawaban_kotor.split("assistant")[-1]

    jawaban_bersih = jawaban_bersih.replace("<|eot_id|>", "").strip()
    return jawaban_bersih, list(set(sumber_referensi))

print("[INFO] Arsitektur RAG Siap!")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

[INFO] Memuat Model Reranker (Cross-Encoder)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Arsitektur RAG Siap!


In [27]:
# Chatboot Advanced RAG
from IPython.display import display, Markdown, clear_output
import warnings
import os

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

print("\n" + "="*50)
print("Chatbot Tim Legal Activated!")
print("Ketik 'Keluar' atau 'exit' untuk selesai!")
print("="*50 + "\n")

while True:
    pertanyaan_user = input(" Pertanyaan Anda: ")

    if pertanyaan_user.lower() in ['keluar', 'exit', 'quit']:
        print("Sesi Ditutup. Kerja bagus!")
        break

    print("Memproses Alur HyDE > Reranker > Threshold Fallback...")

    jawaban_bot, sitasi = advanced_rag_pipeline(pertanyaan_user, threshold_skor=0.1)

    clear_output()

    # tampilkan hasil akhir
    print("\n" + "-"*50)
    display(Markdown(f"** Jawaban Final:**\n{jawaban_bot}"))
    print("\n** Sumber Dokumen Terverifikasi:**")
    for src in sitasi:
        print(src)
    print("="*50 + "\n")


--------------------------------------------------


** Jawaban Final:**
Menurut Pasal 25 dari Peraturan Menteri, pelaksanaan waktu kerja dan jam kerja bagi Pekerja/Buruh yang dipekerjakan pada sektor usaha atau pekerjaan tertentu yang menerapkan waktu kerja lebih dari 8 jam per hari, atau 40 jam per minggu, maka wajib mendapat Upah Lembur.


** Sumber Dokumen Terverifikasi:**
- PP Nomor 35 Tahun 2021.pdf (Halaman 15) [Reranker Score: 0.09]
- PP Nomor 35 Tahun 2021.pdf (Halaman 16) [Reranker Score: 0.04]
- UU Nomor 6 Tahun 2023.pdf (Halaman 723) [Reranker Score: 0.85]



KeyboardInterrupt: Interrupted by user